# Session 19: Forecasting and a first end-to-end model

Forecast malaria cases from the clinic file. Beat the same-month-last-year baseline and report the miss in cases.

**Student version.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Load and sort
Read `clinic_visits.csv` with `month` as a date. Sort by `district` and `month`. Store the table in `clinic`. Count the blank `total_visits` and store the count in `n_blank`.

In [ ]:
import pandas as pd
import numpy as np

# Your turn: read clinic_visits.csv with the month column read as dates
# Your turn: sort by district, then month, into clinic
# Your turn: count the blank total_visits into n_blank


In [ ]:
_missing = [v for v in ('clinic', 'n_blank',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert clinic.shape == (720, 7), 'clinic should have 720 rows and 7 columns. Check the file name and that you kept every row'
assert n_blank == 18, 'n_blank should be 18: count the blank cells in the total_visits column'
print('Task 1 correct')

## Task 2. Seasonal naive for Gulu
For Gulu, forecast every month of 2025 with the same month of 2024. Store the average miss in `gulu_miss`.

In [ ]:
# Your turn: pick Gulu's malaria_cases with month as the index
# Your turn: the forecast for 2025 is the 2024 values
# Your turn: average the gaps, ignoring the sign, into gulu_miss


In [ ]:
_missing = [v for v in ('gulu_miss',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert abs(gulu_miss - 980) < 2, 'gulu_miss should be about 980 cases. Compare the 2025 months with the 2024 months for Gulu only, ignore the sign, then take the mean'
print('Task 2 correct')

## Task 3. Lag features
Inside each district, make `rain_lag1`, `cases_lag1` and `cases_lag12`. Then find the correlation of cases with last month's rain, in `r_lag`. Round to 2 places.

In [ ]:
# Your turn: add the three lag columns, shifting inside each district, not across the whole table
# Your turn: drop rows with no rain_lag1
# Your turn: correlate rain_lag1 with malaria_cases into r_lag


In [ ]:
_missing = [v for v in ('r_lag',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert 0.30 <= r_lag <= 0.45, 'r_lag should be about 0.37. Did you shift inside each district, drop the rows with no rain_lag1, and round to 2 places?'
print('Task 3 correct')

## Task 4. Split by time
Keep rows that have all three lag features. Store them in `ready`. Training is every month before 2025, stored in `train`. Test is 2025, stored in `test`.

In [ ]:
feats = ['cases_lag1', 'cases_lag12', 'rain_lag1']
# Your turn: drop rows missing any feature into ready
# Your turn: train = months before 2025, test = months from 2025


In [ ]:
_missing = [v for v in ('test', 'train',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert len(test) == 144, 'test should hold 144 rows: 12 districts times the 12 months of 2025'
assert train.month.max() < test.month.min(), 'train must end before test starts. Split on the month column'
print('Task 4 correct')

## Task 5. Baseline and model
Work out the average miss in cases for the seasonal naive forecast (`cases_lag12`) and for a straight-line model fitted on `train`. Store them in `naive_mae` and `model_mae`.

In [ ]:
# Your turn: naive_mae = average gap between malaria_cases and cases_lag12 in test
# Your turn: fit coefficients on train with np.linalg.lstsq (add a column of ones)
# Your turn: predict test, then average the gaps into model_mae


In [ ]:
_missing = [v for v in ('model_mae', 'naive_mae',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert 430 <= naive_mae <= 490, 'naive_mae should be near 461: the average gap between malaria_cases and cases_lag12 in test'
assert 300 <= model_mae <= 380, 'model_mae should be near 338. Fit on train only, with a column of ones, and predict test'
assert model_mae < naive_mae, 'The model should beat the baseline, so model_mae should be smaller than naive_mae'
print('Task 5 correct')

## Task 6. A Pipeline
Build a Pipeline with a median filler, a one-hot `district` and a LinearRegression. Fit it on `train`, predict `test` and store the average miss in `pipe_mae`. This cell uses scikit-learn, so your number will be close to 220 but may differ a little.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

cols = feats + ['district']
# Your turn: build prep, a ColumnTransformer that fills gaps in the three number
#            features with their median and one-hot encodes district
# Your turn: build model, a Pipeline of prep followed by a LinearRegression
# Your turn: fit it on train[cols], predict test[cols], and store the average
#            miss in cases in pipe_mae


In [ ]:
_missing = [v for v in ('pipe_mae',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert 150 <= pipe_mae <= 350, 'pipe_mae should be between 150 and 350 cases. Fit on train[cols], predict test[cols], then average the gaps ignoring the sign'
print('Task 6 correct')

## Task 7. Report it
Write three sentences below. How far off is the model, in cases? Does it beat the baseline, and by how much? Name one limit, for example that the test is a single year.